## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 20.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 271.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 272.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 163.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q
!pip install --upgrade -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama
from IPython.display import Markdown

import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'langchain.text_splitter'

## Question Answering using LLM

#### Downloading and Loading the model

In [19]:
# Define the model repository and filename for the Llama-2-13B-chat-GGUF GGUF model.
model_repo = "TheBloke/Llama-2-13B-chat-GGUF"
model_file = "llama-2-13b-chat.Q5_K_M.gguf"

In [20]:
# Download the model
model_path = hf_hub_download(
    repo_id= model_repo,
    filename= model_file
)

llama-2-13b-chat.Q5_K_M.gguf:   0%|          | 0.00/9.23G [00:00<?, ?B/s]

In [21]:
# Initialize the model with the downloaded GGUF file.
# model_path: path to the GGUF model file.
# n_ctx: context window size (determines how much text the model can process at once).
# n_gpu_layers: number of layers to offload to the GPU for acceleration.
# n_batch: batch size for processing.
llm = Llama(
    model_path=model_path,
    n_ctx=5000,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [53]:
def response(query,max_tokens=200,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [8]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
output_1 = response(query_1)

# Print the output
print("Model Output:")
print(output_1)

Llama.generate: prefix-match hit


Model Output:

Sepsis is a life-threatening condition that can arise from an infection, and it is a leading cause of death in hospitals. The management of sepsis in a critical care unit (CCU) requires a systematic approach that includes early recognition, prompt treatment, and ongoing monitoring. Here is a general protocol for managing sepsis in a CCU:

1. Early recognition:
	* Use a standardized screening tool, such as the Sepsis-3 definition, to identify patients who are at risk of developing sepsis.
	* Monitor patients for signs and symptoms of sepsis, such as fever, tachycardia, tachypnea, and altered mental status.
2. Initial assessment:
	* Perform a thorough physical examination to identify any signs of infection or organ dysfunction.
	* Order blood cultures and other


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
output_2 = response(query_2)

# Print the output
print("Model Output:")
print(output_2)

Llama.generate: prefix-match hit


Model Output:


Answer:

Appendicitis is a medical emergency that requires prompt treatment. The most common symptoms of appendicitis include:

1. Severe pain in the abdomen, usually starting near the belly button and then moving to the lower right side of the abdomen.
2. Nausea and vomiting.
3. Loss of appetite.
4. Fever.
5. Abdominal tenderness and guarding (muscle tension).
6. Abdominal swelling.
7. Diarrhea or constipation


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
output_3 = response(query_3)

# Print the output
print("Model Output:")
print(output_3)

Llama.generate: prefix-match hit


Model Output:

Sudden patchy hair loss, also known as alopecia areata, can be caused by a variety of factors. Here are some effective treatments and solutions for addressing this condition:

1. Corticosteroid injections: These injections can help suppress the immune system and promote hair growth. They are usually given every 4-6 weeks and can be effective in promoting hair regrowth.
2. Topical corticosteroids: Over-the-counter or prescription topical corticosteroids can be applied directly to the affected area to reduce inflamm


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
output_4 = response(query_4)

# Print the output
print("Model Output:")
print(output_4)

Llama.generate: prefix-match hit


Model Output:

There are several treatments that may be recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. The specific treatment plan will depend on the location and severity of the injury, as well as the individual's overall health and medical history. Some common treatments for brain injuries include:

1. Medications: To manage symptoms such as pain, inflammation, and anxiety.
2. Rehabilitation therapy: To help regain lost function and improve cognitive, physical, and emotional abilities.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
output_5 = response(query_5)

# Print the output
print("Model Output:")
print(output_5)

Llama.generate: prefix-match hit


Model Output:


A person who has fractured their leg during a hiking trip requires prompt medical attention to ensure proper healing and prevent complications. Here are some necessary precautions and treatment steps:

1. Stop all activity: The first step is to stop all activity and rest the affected leg immediately. This will help prevent further damage and reduce the risk of complications.
2. Assess the severity of the injury: Evaluate the extent of the fracture and determine if there are any other injuries that require attention.
3. Immobilize the leg: Use a splint


## Question Answering using LLM with Prompt Engineering

In [5]:
def build_prompt(role, instructions, query):
    """
    Build a structured prompt for LLM usage.

    :param role: Role assigned to the model (string)
    :param instructions: List of instruction strings
    :param query: User question (string)
    :return: Formatted prompt string
    """

    # Convert instruction list to bullet format
    formatted_instructions = "\n".join(f"- {ins}" for ins in instructions)

    return f"""
You are a {role}.

Instructions:
{formatted_instructions}

User Question:
{query}

Answer:
"""


In [51]:
def response_with_prompt(role,instruction,query,max_tokens=0,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=build_prompt(role,instruction,query),
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
role = "helpful, respectful and honest medical assistant"

instructions = [
    "Always explain in simple terms for a general audience",
    "Do not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content",
    "Ensure responses are socially unbiased and positive in nature"
]

query_2_1 = "What is the protocol for managing sepsis in a critical care unit?"

print(response_with_prompt(
    role=role,
    instruction=instructions,
    query=query_2_1
))

Llama.generate: prefix-match hit


Hello! As a helpful medical assistant, I'm here to provide you with information on managing sepsis in a critical care unit. Sepsis is a serious and potentially life-threatening condition that can arise from an infection, so it's important to follow established protocols to ensure the best possible outcomes for our patients.

In our critical care unit, we follow the Surviving Sepsis Campaign (SSC) guidelines, which are evidence-based recommendations for the management of sepsis. These guidelines emphasize early recognition and aggressive treatment of sepsis, including administration of antibiotics, fluid resuscitation, and monitoring of vital signs.

Our protocol for managing sepsis includes:

1. Early identification and reporting of sepsis: Our team is trained to recognize the signs and symptoms of sepsis, such as fever, tachycardia, tachypnea, and altered mental status. We report any suspected cases of sepsis to the attending physician immediately.
2. Rapid administration of antibioti

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [15]:
# temperature set to 1
role = "helpful, respectful and honest medical assistant"

instructions = [
    "Respond briefly and clearly in Shakespearean language."
]

query_2_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

print(response_with_prompt(
    role=role,
    instruction=instructions,
    max_tokens=0,
    temperature=0,
    top_p=0.95,
    top_k=5,
    query=query_2_2
))

Llama.generate: prefix-match hit


O, fair user, with symptoms of appendicitis thou dost beset!
The common signs of this woeful plight include fierce pains in the side,
Nausea and vomiting, fever and chill, doth make thee feel quite ill.

Alas, medicine cannot cure this malady, for it doth not heal
The inflamed appendix within thee, which doth cause such pain and steal
Thy vital strength away. Surgery is the only way to set it right,
A procedure called appendectomy, wherein the Appendix doth take flight.

There be two types of surgeries, O user, one with a small incision made,
And another with a larger cut, which doth provide more access to the aid.
Both procedures successful in removing the inflamed part,
And bringing thee relief from pain and restoring thy healthy heart.

Choose the one that suits thee best, O user, and may thy recovery be swift!


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [16]:
# top_k set to 5
role = "helpful, respectful and honest medical assistant"

instructions = []

query_2_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

print(response_with_prompt(
    role=role,
    instruction=instructions,
    temperature=0,
    top_k=5,
    query=query_2_3
))

Llama.generate: prefix-match hit


Thank you for reaching out with your concerns about sudden patchy hair loss. There are several potential causes and effective treatments for this condition.

Possible Causes:

1. Hormonal Imbalance: Hormonal fluctuations, particularly an excess of dihydrotestosterone (DHT), can lead to hair loss. This is a common cause of patchy hair loss in both men and women.
2. Autoimmune Disorders: Conditions like alopecia areata, lupus, or rheumatoid arthritis can cause sudden hair loss due to an immune system attack on the hair follicles.
3. Nutritional Deficiencies: Iron deficiency, zinc deficiency, or a lack of essential vitamins like biotin and vitamin D can contribute to hair loss.
4. Skin Conditions: Fungal infections like ringworm, eczema, or psoriasis can cause patchy hair loss due to inflammation and scalp irritation.
5. Physical Trauma: Injury to the scalp or hair follicles can lead to sudden hair loss.

Effective Treatments:

1. Medications: Minoxidil (Rogaine) and finasteride (Propecia

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [17]:
role = "helpful, respectful and honest medical assistant"

instructions = [

"Question: What are the common symptoms and treatments for pulmonary embolism? Answer: Common symptoms of pulmonary embolism include sudden shortness of breath, chest pain that worsens with breathing or coughing, rapid heart rate, rapid breathing, anxiety, coughing (sometimes with blood), sweating, and fainting. Treatment typically involves anticoagulant medications to prevent further clots, and sometimes thrombolytics to dissolve existing clots. In severe cases, surgical embolectomy or catheter-directed treatments may be necessary.",

"Question: Can you provide the trade names of medications used for treating hypertension? Answer: Some common trade names for medications used to treat hypertension include Prinivil, Zestril (Lisinopril), Norvasc (Amlodipine), Cozaar (Losartan), Diovan (Valsartan), Toprol XL, Lopressor (Metoprolol), and Tenormin (Atenolol).",

"Question: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function? Answer:"
]

query_2_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

print(response_with_prompt(
    role=role,
    instruction=instructions,
    max_tokens=0,
    query=query_2_4
))

Llama.generate: prefix-match hit


Treatment options for a person who has sustained a physical injury to brain tissue will depend on the severity and location of the injury, as well as the individual's overall health. Some common treatments may include medications to manage symptoms such as pain, anxiety, or depression; physical therapy to improve motor function and mobility; occupational therapy to help with daily activities and cognitive function; speech therapy to address communication and language difficulties; and rehabilitation programs to help regain lost skills and abilities. In some cases, surgery may be necessary to relieve pressure on the brain or repair damaged tissue. Additionally, supportive care such as nutritional support, hydration, and pain management may be provided to ensure the individual's comfort and well-being. The goal of treatment is to help the individual recover as much function as possible and improve their quality of life.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [7]:
role = "helpful, respectful and honest medical assistant"

instructions = [
 "Think step-by-step to determine the necessary precautions, treatment steps, and considerations for care and recovery for a person who has fractured their leg during a hiking trip. Consider the immediate actions to take at the injury site, the subsequent medical treatment, and the long-term recovery process.",

]

query_2_5 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

print(response_with_prompt(
    role=role,
    instruction=instructions,
    max_tokens=0,
    temperature=0,
    top_p=0.95,
    top_k=50,
    query=query_2_5
))

If a person has sustained an injury to brain tissue that has resulted in temporary or permanent impairment of brain function, the recommended treatment will depend on the severity and location of the injury. Here are some possible treatments that may be considered:

1. Medications: Depending on the type and severity of the injury, medications such as pain relievers, anti-seizure drugs, or antidepressants may be prescribed to manage symptoms and promote healing.
2. Rehabilitation therapy: Physical, occupational, and speech therapy may be necessary to help the patient regain lost function and improve cognitive and physical abilities.
3. Surgery: In some cases, surgery may be required to relieve pressure on the brain or repair damaged blood vessels.
4. Lifestyle changes: Patients with brain injuries may need to make significant lifestyle changes, such as avoiding activities that exacerbate symptoms, taking regular breaks to rest, and modifying their diet to improve nutrition and energy le

## Data Preparation for RAG

### Loading the Data

In [2]:
med_diag_pdf_path = "/content/medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(med_diag_pdf_path)
medical_diagnosis_manual = pdf_loader.load()

NameError: name 'PyMuPDFLoader' is not defined

In [3]:
llm = Llama(
    model_path=model_path,
    n_ctx=10000,
    n_gpu_layers=38,
    n_batch=512
)

NameError: name 'Llama' is not defined

### Data Overview

#### Checking the first 5 pages

In [27]:
for doc in medical_diagnosis_manual[:5]:
    print(f"\nPage Number: {doc.metadata.get('page')}\n")
    print(doc.page_content)


Page Number: 0

anoopvijayan@yahoo.com
DLRSOVW8C6
This file is meant for personal use by anoopvijayan@yahoo.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Page Number: 1

anoopvijayan@yahoo.com
DLRSOVW8C6
This file is meant for personal use by anoopvijayan@yahoo.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Page Number: 2

Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ........................................................................................................................

#### Checking the number of pages

In [6]:
print("Total pages:", len(medical_diagnosis_manual))

Total pages: 4114


### Data Chunking

In [4]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)
document_chunks = pdf_loader.load_and_split(text_splitter)
print("Total chunks:",len(document_chunks))

Total chunks: 8469


### Embedding

In [5]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/670M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [6]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [28]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True



*   The embedding model provides a fixed-length vector for any number of chunks.
*   This is necessary because we want to compare them for similarity.

### Vector Database

In [7]:
out_dir = 'med_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [8]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [9]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [10]:
vectorstore.similarity_search("Pathophysiology",k=3)

[Document(metadata={'creationDate': 'D:20120615054440Z', 'moddate': '2026-03-03T15:59:55+00:00', 'format': 'PDF 1.7', 'trapped': '', 'file_path': '/content/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'creationdate': '2012-06-15T05:44:40+00:00', 'modDate': 'D:20260303155955Z', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'source': '/content/medical_diagnosis_manual.pdf', 'page': 242, 'keywords': '', 'author': '', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)'}, page_content='Pathophysiology\nCrohn\'s disease begins with crypt inflammation and abscesses, which progress to tiny focal aphthoid\nulcers. These mucosal lesions may develop into deep longitudinal and transverse ulcers with intervening\nmucosal edema, creating a characteristic cobblestoned appearance to the bowel.\nTransmural spread of inflammation leads to lymphedema and thickening of the bowel wall and mesentery.\nMesenteric fat typ

### Retriever

In [15]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [29]:
rel_docs = retriever.get_relevant_documents("What are the Major Differences Between Pediatric and Adult CPR?")
rel_docs

[Document(metadata={'total_pages': 4114, 'subject': '', 'page': 2418, 'moddate': '2026-03-03T15:59:55+00:00', 'creator': 'Atop CHM to PDF Converter', 'file_path': '/content/medical_diagnosis_manual.pdf', 'author': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'keywords': '', 'trapped': '', 'modDate': 'D:20260303155955Z', 'creationdate': '2012-06-15T05:44:40+00:00', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'source': '/content/medical_diagnosis_manual.pdf', 'format': 'PDF 1.7', 'creationDate': 'D:20120615054440Z'}, page_content='modified Pittsburgh Outcome Categories Scale reflects cerebral and overall performance (see\nTable 223-4).\nMajor Differences Between Pediatric and Adult CPR\nPrearrest: Bradycardia in a distressed child is a sign of impending cardiac arrest . Neonates, infants,\nand young children are more likely to develop bradycardia caused by hypoxemia, whereas older children\ninitially tend to have tachycardia. An infant or child wi

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [48]:
qna_system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [46]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [47]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [1]:
user_input = "Question What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input))

NameError: name 'generate_rag_response' is not defined

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [35]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
Markdown(generate_rag_response(user_input))

Llama.generate: prefix-match hit


Please answer based on the provided context.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

### Fine-tuning

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message  = ""

In [ ]:
relevance_rater_system_message = ""

In [ ]:
user_message_template = ""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>
___